In [1]:
%load_ext autoreload
%autoreload 2
%reload_ext autoreload

import nest_asyncio
nest_asyncio.apply()

import plotly.express as px
import plotly.graph_objects as go
import plotly.io as pio
pio.renderers.default = "vscode"            

import matplotlib.pyplot as plt
import matplotlib.pylab as pylab
import matplotlib.dates as mdates
plt.style.use('ggplot')
params = {'legend.fontsize': 'x-large',
        'figure.figsize': (12, 8),
        'axes.labelsize': 'x-large',
        'axes.titlesize':'x-large',
        'xtick.labelsize':'x-large',
        'ytick.labelsize':'x-large'}
pylab.rcParams.update(params)

import pandas as pd
import numpy as np

import datetime
import pytz
NY_tz = pytz.timezone("America/New_York") 
CHI_tz = pytz.timezone("America/Chicago") 
UTC_tz = pytz.timezone("UTC") 

In [164]:
import rateslib as rl
import QuantLib as ql
import requests
from pandas.tseries.offsets import BDay

from MDP.IRSwaps.IRSwapsMDP import IRSwapsMDP

In [159]:
stir_curve_mdp, curve = IRSwapsMDP(source="SDR_INTRADAY-rl_usd_ois_stir_q12x12"), "USD-FEDFUNDS"
# stir_curve_mdp, curve = IRSwapsMDP(source="SDR_INTRADAY-rl_usd_sofr_stir_q13x10"), "USD-SOFR-1D"

In [160]:
def get_tbill_info(as_of: datetime.date = None) -> pd.DataFrame:
    if as_of is None:
        as_of = datetime.date.today()
    base = "https://api.fiscaldata.treasury.gov/services/api/fiscal_service/v1/accounting/od/auctions_query?"
    start_pad = as_of - datetime.timedelta(days=365 * 30)  # extra room for boundary logic
    f_dates = f"record_date:lte:{as_of.strftime('%Y-%m-%d')},record_date:gte:{start_pad.strftime('%Y-%m-%d')},maturity_date:gte:{as_of.strftime('%Y-%m-%d')}"
    f_tips = "inflation_index_security:eq:No"
    f_type = "security_type:eq:Bill"
    params = f"filter={f_dates},{f_tips},{f_type}&" "sort=-record_date&" "page[size]=5000"
    url = base + params
    resp = requests.get(url)
    resp.raise_for_status()
    data = resp.json().get("data", [])
    df = pd.DataFrame(data)
    df = df[df["frn_index_determination_rate"] == "null"]  # no frns

    for col in ["auction_date", "issue_date", "maturity_date"]:
        if col in df.columns:
            df[col] = pd.to_datetime(df[col], errors="coerce")
            
    df = df.sort_values(by=["issue_date"])
    df = df.drop_duplicates(subset=["cusip"], keep="last")
    df = df[["cusip", "auction_date", "issue_date", "maturity_date", "price_per100", "high_investment_rate", "security_term", "original_security_term"]]

    dc = ql.ActualActual(ql.ActualActual.Actual365)
    ql.Settings.instance().evaluationDate = ql.Date(as_of.day, as_of.month, as_of.year)

    def _yf(mdate: datetime.date) -> float:
        if pd.isna(mdate):
            return float("nan")
        d1 = ql.Date(as_of.day, as_of.month, as_of.year)
        d2 = ql.Date(mdate.day, mdate.month, mdate.year)
        return dc.yearFraction(d1, d2)

    def _dcnt(mdate: datetime.date) -> int:
        if pd.isna(mdate):
            return pd.NA
        d1 = ql.Date(as_of.day, as_of.month, as_of.year)
        d2 = ql.Date(mdate.day, mdate.month, mdate.year)
        return int(dc.dayCount(d1, d2))

    df["ttm_y"] = df["maturity_date"].map(_yf)
    df["ttm_d"] = df["maturity_date"].map(_dcnt)
    df["rank"] = df.groupby("original_security_term")["issue_date"].rank(method="first", ascending=False).astype(int) - 1

    return df


def fetch_fedinvest_price(as_of: datetime.date, cusips: list[str]):
    as_of = as_of + BDay(1)
    url = "https://treasurydirect.gov/GA-FI/FedInvest/selectSecurityPriceDate"
    payload = {
        "priceDate.month": as_of.month,
        "priceDate.day": as_of.day,
        "priceDate.year": as_of.year,
        "submit": "Show+Prices",
    }
    res = requests.post(url, data=payload)
    res.raise_for_status()
    tables = pd.read_html(res.content, header=0)
    df = tables[0]
    df.columns = df.columns.str.lower()
    df = df.rename(
        columns={
            "buy": "offer_price",
            "security type": "type",
            "rate": "coupon",
            "sell": "bid_price",
        }
    ).drop(columns=["end of day", "call date", "type", "coupon", "maturity date"])

    for col in ["offer_price", "bid_price"]:
        if col in df:
            df[col] = df[col].astype(str).str.replace(",", "", regex=False).str.replace("$", "", regex=False)
            df[col] = pd.to_numeric(df[col], errors="coerce")

    prices = df[["offer_price", "bid_price"]].copy()
    prices = prices.replace(0.0, np.nan)  # treat 0 as missing
    df["mid_price"] = prices.mean(axis=1, skipna=True)
    df["cusip"] = df["cusip"].astype(str).str.strip().str.upper()
    df = df[["cusip", "mid_price"]]
    return df[df["cusip"].isin(cusips)]


def price_tbill(issue, maturity, price, settle):
    return rl.Bill(issue, maturity, spec="us_gbb").simple_rate(price, settle)
    # return rl.Bill(issue, maturity, spec="us_gbb").discount_rate(price, settle)


def price_tbill_mms(effective, maturity, curve_handle):
    return rl.IRS(effective=effective, termination=maturity, spec="usd_irs_lt_2y", curves=curve_handle, notional=1).rate().real

In [161]:
as_of = NY_tz.localize(datetime.datetime(2025, 9, 24, 17, 00)) 
settle = as_of.date() + BDay(1)

tbills_df = get_tbill_info(as_of.date())
tbill_prices = fetch_fedinvest_price(as_of=as_of.date(), cusips=tbills_df["cusip"])
tbills_df = pd.merge(left=tbills_df, right=tbill_prices, on="cusip")
tbills_df["tbill_rate"] = tbills_df.apply(lambda x: price_tbill(x["issue_date"], x["maturity_date"], x["mid_price"], settle), axis=1)

stir_curve_handle = stir_curve_mdp._get_curve(curve_name=curve, timestamp=as_of)
tbills_df["swap_rate"] = tbills_df.apply(lambda x: price_tbill_mms(settle, x["maturity_date"], stir_curve_handle.handle()), axis=1)
tbills_df["mmss (bps)"] = (tbills_df["swap_rate"] - tbills_df["tbill_rate"]) * 100

tbills_df = tbills_df.sort_values(by="maturity_date")

In [162]:
import sys
sys.path.append(r"C:\Users\chris\clee\project-oasis")
from public.RVpy.ust_viz import plot_usts 
from private.rvcore.core.CurveBuilding.utils.Interpolation.GeneralCurveInterpolator import GeneralCurveInterpolator

In [163]:
loess_spline = GeneralCurveInterpolator(x=tbills_df["ttm_d"], y=tbills_df["mmss (bps)"]).loess_interpolation(frac=0.5, it=100, return_func=True)

plot_usts(
    curve_set_df=tbills_df,
    ttm_col="ttm_d",
    custom_x_axis="Days",
    ytm_col="mmss (bps)",
    custom_y_axis="MMSS (bps)",
    label_col="original_security_term",
    cusip_col="cusip",
    hover_data=["issue_date", "maturity_date"],
    splines=[(loess_spline, "LOESS Spline")],
    title=f"TBill MMSS {stir_curve_handle.id()} | {as_of}",
    spline_ub=360,
    ignore_otr=True
)